In [ ]:
!npm install -g localtunnel
!pip install -q pypdf
!pip install -q langchain-text-splitters
!pip install -q langchain langchain-community langchain-groq \
    faiss-cpu sentence-transformers PyPDF2 \
    streamlit pyngrok python-dotenv tiktoken


In [ ]:
import os
from google.colab import userdata

# Set your Groq API key securely
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Quick check
import groq
client = groq.Groq()
print("Groq connected ✓")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os

# File is already uploaded, load it directly
filename = "Physical Chemistry of Metallurgical Processes - 2016 - Shamsuddin (1).pdf"

loader = PyPDFLoader(filename)
documents = loader.load()
print(f"✓ Loaded: {filename} ({len(documents)} pages)")

# Semantic chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\n--- Sample Chunk ---")
print(chunks[0].page_content)
print(f"\nSource: {chunks[0].metadata}")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import time

# Load HuggingFace embedding model (free, no API key needed)
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("✓ Embedding model loaded")

# Build FAISS index from chunks
print(f"\nEmbedding {len(chunks)} chunks into FAISS index...")
start = time.time()
vectorstore = FAISS.from_documents(chunks, embeddings)
elapsed = time.time() - start
print(f"✓ FAISS index built in {elapsed:.1f}s")

# Save index to disk
vectorstore.save_local("faiss_index")
print("✓ Index saved to faiss_index/")

# Quick retrieval test
print("\n--- Retrieval Test ---")
query = "What is the role of temperature in metallurgical processes?"
results = vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(results):
    print(f"\nResult {i+1} (page {doc.metadata.get('page', '?')}):")
    print(doc.page_content[:200])

In [ ]:
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import os

# Load Groq LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2,
    api_key=os.environ["GROQ_API_KEY"]
)
print("✓ Groq LLM loaded")

# Load FAISS index
vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)
print("✓ Retriever ready")

# Prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant that answers questions based strictly on the provided context.
Always cite the source by mentioning page numbers when available.
If the answer is not in the context, say 'I could not find this in the document.'

Context:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}")
])

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(
        f"[Page {doc.metadata.get('page', '?')}]: {doc.page_content}"
        for doc in docs
    )

# Conversation memory
chat_history = []

def ask(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "chat_history": chat_history,
        "question": question
    })
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    return answer, docs

print("✓ RAG chain ready\n")

# Test
answer, sources = ask("What is the role of temperature in metallurgical processes?")
print("Answer:\n", answer)
print("\nSources:")
for doc in sources:
    print(f"  → Page {doc.metadata.get('page', '?')}: {doc.page_content[:100]}...")

In [ ]:
app_code = '''
import streamlit as st
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import os

# Page config
st.set_page_config(page_title="RAG Document Assistant", page_icon="📄", layout="wide")
st.title("📄 RAG Document Assistant")
st.caption("Ask questions about your PDF — answers grounded in the document with page citations.")

# Load API key
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

# Initialize session state
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []
if "messages" not in st.session_state:
    st.session_state.messages = []

@st.cache_resource
def load_chain():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True}
    )
    vectorstore = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True
    )
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )
    llm = ChatGroq(
        model="llama-3.3-70b-versatile",
        temperature=0.2,
        api_key=GROQ_API_KEY
    )
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a helpful assistant that answers questions based strictly on the provided context.
Always cite the source by mentioning page numbers when available.
If the answer is not in the context, say: I could not find this in the document.

Context:
{context}"""),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}")
    ])
    return retriever, llm, prompt

def format_docs(docs):
    return "\\n\\n".join(
        f"[Page {doc.metadata.get(\'page\', \'?\')}]: {doc.page_content}"
        for doc in docs
    )

def ask(question, retriever, llm, prompt):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({
        "context": context,
        "chat_history": st.session_state.chat_history,
        "question": question
    })
    st.session_state.chat_history.append(HumanMessage(content=question))
    st.session_state.chat_history.append(AIMessage(content=answer))
    return answer, docs

# Load chain
with st.spinner("Loading model and index..."):
    retriever, llm, prompt = load_chain()

# Display chat history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Chat input
if question := st.chat_input("Ask a question about your document..."):
    # Show user message
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    # Get answer
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            answer, sources = ask(question, retriever, llm, prompt)
        st.markdown(answer)

        # Show sources
        with st.expander("📚 Sources"):
            for doc in sources:
                page = doc.metadata.get("page", "?")
                st.markdown(f"**Page {page}:**")
                st.caption(doc.page_content[:300] + "...")

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✓ app.py written")

In [ ]:
!ngrok authtoken 3EzNdHWxMo3HHAKoga1HAorHOt4_6gviDoNm573ZkgZqbUcQp

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok

os.environ["GROQ_API_KEY"] = os.environ["GROQ_API_KEY"]

# Kill existing processes
os.system("pkill -f streamlit")
ngrok.kill()
time.sleep(2)

# Start Streamlit
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.headless", "true",
    "--server.enableCORS", "false"
])
time.sleep(5)
print("✓ Streamlit started")

# Create tunnel
public_url = ngrok.connect(8501)
print(f"✓ App is live at: {public_url}")

In [ ]:
import shutil
shutil.make_archive("faiss_index", "zip", "faiss_index")
print("✓ faiss_index.zip created")